### <span style = "color:orange">1. numba<span>

In [ ]:
# from temp import benchmark_numba

In [ ]:
# # Ví dụ test
# benchmark_numba(samples=300_000*13, lib='python')
# benchmark_numba(samples=300_000*13, lib='numpy')
# benchmark_numba(samples=300_000*13, lib='pandas')

### <span style = "color:orange">2. numpy vs cupy<span>

Requierments:
- must learn cupy syntax

### <span style = "color:orange">3.1 pandas vs cudf<span>

Requirement:
- compute capability 7.0+
  

<a href="https://youtu.be/6zM6DqHB9-g?si=1d6qUkxk1kkCXJNb">150x Pandas Speed-Up: cuDF in Python</a>

<a href="https://www.youtube.com/watch?v=OnYGtKQT-rU">Pandas Dataframes on your GPU w/ CuDF</a>

### <span style = "color:orange">3.2. pandas vs polars<span>

<a href="https://viblo.asia/p/polars-thu-vien-xu-ly-du-lieu-dataframe-nhanh-hon-ca-pandas-oK9VyQjOVQR">Polars - thư viện xử lý dữ liệu DataFrame nhanh hơn cả Pandas!!!!</a>

### <span style = "color:orange">4. matplotlib<span>

### <span style = "color:orange">5. sklearn vs cuml<span>

faster <~> decrease accuracy

### <span style = "color:orange">6. tf-cpu vs tf-gpu<span>

faster <~> may decrease accuracy

In [ ]:
import tensorflow as tf

In [ ]:
# import numpy as np
# from temp import benchmark_tf

In [ ]:
list(tf.config.list_physical_devices("CPU")), list(tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
import numpy as np
import time

def create_model(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(64, activation='relu', input_shape=input_shape),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

def benchmark_tf(x, y, epochs=5, batch_size=64):
    results = {}

    configs = [
        ("CPU + Numpy", "/CPU:0", False),
        ("GPU + Numpy", "/GPU:0", False),
        ("CPU + from_tensor_slices", "/CPU:0", True),
        ("GPU + from_tensor_slices", "/GPU:0", True),
    ]

    for name, device, use_dataset in configs:
        with tf.device(device):
            print("Currently using:",device)
            if use_dataset:
                dataset = tf.data.Dataset.from_tensor_slices((x, y))
                dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
                fit_args = {"x": dataset, "epochs": epochs, "verbose": 1}
            else:
                fit_args = {"x": x, "y": y, "batch_size": batch_size, "epochs": epochs, "verbose": 1}

            model = create_model((x.shape[1],), len(np.unique(y)))

            start = time.time()
            model.fit(**fit_args)
            elapsed = time.time() - start

            results[name] = elapsed
            print(f"{name}: {elapsed:.2f} giây")

    return results

# Test
x = np.random.rand(280000*6, 13).astype(np.float32)
y = np.random.randint(0, 10, size=(280000*6,))
benchmark_tf(x, y, epochs=5, batch_size=1024)